# User-User Collaborative Filtering

## Learning Objectives

1. **Implement** Pearson correlation for user similarity
2. **Select** the k nearest neighbor users for a target user
3. **Predict** ratings using weighted average of neighbors
4. **Discuss** the scalability and accuracy trade-offs vs item-item CF

## The User-User CF Idea

**Assumption:** if two users $u$ and $v$ have rated many items similarly, they likely have similar tastes.

**Algorithm:**
1. For target user $u$ and item $i$ (unrated by $u$):
2. Find users who rated item $i$ → neighbors
3. Weight each neighbor's rating by their similarity to $u$
4. Predicted rating: weighted average

$$\hat{r}_{u,i} = \bar{r}_u + \frac{\sum_{v \in N} \text{sim}(u,v) (r_{v,i} - \bar{r}_v)}{\sum_{v \in N} |\text{sim}(u,v)|}$$

Subtracting $\bar{r}_v$ corrects for different rating scales (some users always rate high).

In [1]:
import math

def pearson(ratings_u, ratings_v):
    """Pearson correlation between two rating dictionaries."""
    common = [item for item in ratings_u if item in ratings_v]
    if len(common) < 2: return 0.0
    ru = [ratings_u[i] for i in common]
    rv = [ratings_v[i] for i in common]
    mean_u = sum(ru)/len(ru); mean_v = sum(rv)/len(rv)
    num = sum((a-mean_u)*(b-mean_v) for a,b in zip(ru,rv))
    den = math.sqrt(sum((a-mean_u)**2 for a in ru)) * math.sqrt(sum((b-mean_v)**2 for b in rv))
    return num/den if den > 0 else 0.0

# Utility matrix (rows=users, cols=items); None = not rated
ratings = {
    'Alice': {'M1':5,'M2':3,'M3':4,'M4':4},
    'Bob':   {'M1':3,'M2':1,'M3':2,'M4':3,'M5':3},
    'Carol': {'M1':4,'M2':3,'M3':4,'M5':5},
    'Dave':  {'M2':1,'M3':2,'M4':3,'M5':4},
    'Eve':   {'M1':1,'M3':5,'M4':2},
}

target_user = 'Alice'
target_item = 'M5'  # Alice hasn't rated this

# Compute similarity to all other users
sims = {}
for user, r in ratings.items():
    if user != target_user and target_item in r:
        sims[user] = pearson(ratings[target_user], r)

print(f"Similarities to {target_user}:")
for u, s in sorted(sims.items(), key=lambda x:-x[1]):
    print(f"  {u}: {s:.4f}")

# Predict rating
mean_alice = sum(ratings[target_user].values()) / len(ratings[target_user])
print(f"""
Alice's mean rating: {mean_alice:.3f}""")

num = sum(sims[u]*(ratings[u][target_item]-sum(ratings[u].values())/len(ratings[u]))
          for u in sims if sims[u] != 0)
den = sum(abs(sims[u]) for u in sims if sims[u] != 0)
pred = mean_alice + num/den if den > 0 else mean_alice
print(f"Predicted rating for {target_user} on {target_item}: {pred:.3f}")

Similarities to Alice:
  Carol: 0.8660
  Dave: 0.8660
  Bob: 0.8528

Alice's mean rating: 4.000
Predicted rating for Alice on M5: 5.036


In [2]:
def mean_rating(ratings_dict):
    return sum(ratings_dict.values())/len(ratings_dict) if ratings_dict else 0

def predict_rating(target_user, target_item, ratings, k=3):
    """Predict rating using k nearest neighbors."""
    # Users who have rated the target item
    neighbors = {u: pearson(ratings[target_user], r)
                 for u, r in ratings.items()
                 if u != target_user and target_item in r}
    # Top-k by similarity
    top_k = sorted(neighbors.items(), key=lambda x:-x[1])[:k]
    if not top_k: return mean_rating(ratings[target_user])
    mean_u = mean_rating(ratings[target_user])
    num = sum(sim*(ratings[v][target_item]-mean_rating(ratings[v])) for v,sim in top_k)
    den = sum(abs(sim) for _,sim in top_k)
    return mean_u + num/den if den > 0 else mean_u

# Predict all unrated items for Alice
print("Predictions for Alice (unrated items):")
all_items = set(i for r in ratings.values() for i in r)
unrated = all_items - set(ratings['Alice'].keys())
for item in sorted(unrated):
    pred = predict_rating('Alice', item, ratings)
    print(f"  {item}: {pred:.3f}")

Predictions for Alice (unrated items):
  M5: 5.036


## Scalability Issue

For $m$ users and $n$ items:
- Similarity computation: $O(m^2 \cdot n)$ to compute all pairwise similarities
- Per-recommendation: $O(m)$ to find neighbors

At Netflix scale (500K users): $O(500K^2) = 2.5 \times 10^{11}$ similarity computations.

**Solutions:**
1. **Item-item CF:** similarities between $n \ll m$ items are cheaper and more stable
2. **Pre-computation:** compute and store top-K neighbors offline; refresh periodically
3. **Approximate methods:** LSH, clustering to reduce candidate sets

Item-item CF is usually preferred at scale because item catalogs grow slower than user bases, and item similarities are more stable than user similarities.